# 02: 予測表生成

各回号に対して4パターン（A1/A2/B1/B2）の予測表を生成します。

## 目的
- 各回号に対して4パターン（A1/A2/B1/B2）の予測表を生成
- 予測表の検証と可視化
- 特徴量エンジニアリングの準備

## パターン定義
- **A1**: 欠番補足あり（0〜9全追加）+ 中心0配置なし
- **A2**: 欠番補足あり（0〜9全追加）+ 中心0配置あり
- **B1**: 欠番補足なし（0のみ追加）+ 中心0配置なし
- **B2**: 欠番補足なし（0のみ追加）+ 中心0配置あり


In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

# プロジェクトルートのパスを設定
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'

# 予測表生成モジュールをインポート
import sys
sys.path.append(str(PROJECT_ROOT / 'notebooks'))
from chart_generator import (
    load_keisen_master,
    generate_chart,
    Pattern,
    Target
)

print(f"プロジェクトルート: {PROJECT_ROOT}")
print(f"データディレクトリ: {DATA_DIR}")


In [ ]:
# データの読み込み
# 学習用データセット（01_data_preparation.ipynbで作成）
train_csv_path = DATA_DIR / 'train_data_100.csv'

if train_csv_path.exists():
    train_df = pd.read_csv(train_csv_path)
    print(f"学習用データセット: {len(train_df)}回分")
else:
    # なければ過去実績データから読み込む
    csv_path = DATA_DIR / 'past_results.csv'
    df = pd.read_csv(csv_path)
    df = df.sort_values('round_number', ascending=False).reset_index(drop=True)
    
    # 基準回号（6758）からさかのぼって100回分を取得
    BASE_ROUND = 6758
    base_idx = df[df['round_number'] == BASE_ROUND].index[0]
    train_df = df.iloc[base_idx:base_idx+100].copy()
    train_df = train_df.sort_values('round_number', ascending=True).reset_index(drop=True)
    print(f"学習用データセット: {len(train_df)}回分（直接読み込み）")

print(f"回号範囲: {train_df['round_number'].min()} 〜 {train_df['round_number'].max()}")

# 罫線マスターデータの読み込み
keisen_master = load_keisen_master(DATA_DIR)
print("\n罫線マスターデータの読み込み完了")


In [ ]:
# テスト: 1つの回号で4パターンの予測表を生成
test_round = 6758
test_target: Target = 'n3'
patterns: List[Pattern] = ['A1', 'A2', 'B1', 'B2']

print(f"テスト回号: {test_round}")
print(f"対象: {test_target}")
print(f"パターン: {patterns}\n")

for pattern in patterns:
    try:
        grid, rows, cols = generate_chart(
            train_df, keisen_master, test_round, pattern, test_target
        )
        
        print(f"\n=== パターン {pattern} ===")
        print(f"行数: {rows}, 列数: {cols}")
        
        # グリッドの表示（1-indexedなので、1から開始）
        print("\n予測表:")
        for row in range(1, rows + 1):
            row_data = []
            for col in range(1, cols + 1):
                val = grid[row][col]
                row_data.append(str(val) if val is not None else '.')
            print(f"  行{row}: {' '.join(row_data)}")
        
    except Exception as e:
        print(f"\nパターン {pattern} の生成エラー: {e}")


In [ ]:
# 全回号×4パターン×2対象（N3/N4）の予測表を生成
# 注意: これは時間がかかるので、必要に応じてサンプルサイズを調整してください

# サンプル: 最初の10回分のみでテスト
SAMPLE_SIZE = 10  # 全データで実行する場合は len(train_df) に変更
sample_df = train_df.head(SAMPLE_SIZE).copy()

print(f"サンプルサイズ: {len(sample_df)}回分")
print(f"回号範囲: {sample_df['round_number'].min()} 〜 {sample_df['round_number'].max()}")

# 結果を格納するリスト
generated_charts = []

patterns: List[Pattern] = ['A1', 'A2', 'B1', 'B2']
targets: List[Target] = ['n3', 'n4']

for _, row in sample_df.iterrows():
    round_number = row['round_number']
    
    for target in targets:
        for pattern in patterns:
            try:
                grid, rows, cols = generate_chart(
                    train_df, keisen_master, round_number, pattern, target
                )
                
                generated_charts.append({
                    'round_number': round_number,
                    'target': target,
                    'pattern': pattern,
                    'grid': grid,
                    'rows': rows,
                    'cols': cols
                })
                
            except Exception as e:
                print(f"エラー: 回号={round_number}, 対象={target}, パターン={pattern}: {e}")

print(f"\n生成された予測表数: {len(generated_charts)}")
print(f"期待値: {len(sample_df)}回 × {len(targets)}対象 × {len(patterns)}パターン = {len(sample_df) * len(targets) * len(patterns)}")


In [ ]:
# 予測表の検証
# 1. すべてのセルが埋まっているか確認
# 2. 予測表のサイズが正しいか確認

print("=== 予測表の検証 ===\n")

validation_errors = []

for chart in generated_charts:
    round_number = chart['round_number']
    target = chart['target']
    pattern = chart['pattern']
    grid = chart['grid']
    rows = chart['rows']
    cols = chart['cols']
    
    # 1. すべてのセルが埋まっているか確認
    empty_cells = []
    for row in range(1, rows + 1):
        for col in range(1, cols + 1):
            if grid[row][col] is None:
                empty_cells.append((row, col))
    
    if empty_cells:
        validation_errors.append({
            'round_number': round_number,
            'target': target,
            'pattern': pattern,
            'error': f'空のセルがあります: {empty_cells[:5]}...' if len(empty_cells) > 5 else f'空のセルがあります: {empty_cells}'
        })
    
    # 2. 予測表のサイズが正しいか確認
    if rows % 2 != 0:
        validation_errors.append({
            'round_number': round_number,
            'target': target,
            'pattern': pattern,
            'error': f'行数が奇数です: {rows}'
        })
    
    if cols != 8:
        validation_errors.append({
            'round_number': round_number,
            'target': target,
            'pattern': pattern,
            'error': f'列数が8ではありません: {cols}'
        })

if validation_errors:
    print(f"検証エラー: {len(validation_errors)}件")
    for error in validation_errors[:10]:  # 最初の10件のみ表示
        print(f"  回号={error['round_number']}, 対象={error['target']}, パターン={error['pattern']}: {error['error']}")
else:
    print("すべての予測表が正常に生成されました！")


## 次のステップ

予測表の生成が完了しました。次のNotebook (`03_feature_engineering.ipynb`) で、特徴量エンジニアリングを実装します。
